# Task 1: Problem & Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Spark Session

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
    .appName("HIGGS_BigData_Project") \
    .getOrCreate()

2. Load Dataset

In [ ]:
df = spark.read.csv(
    "/content/drive/MyDrive/HIGGS.csv.gz",
    header=False,
    inferSchema=True
)

3. Rename Columns

In [ ]:
columns = [
"label","lepton_pt","lepton_eta","lepton_phi",
"missing_energy","missing_energy_phi",
"jet1_pt","jet1_eta","jet1_phi","jet1_btag",
"jet2_pt","jet2_eta","jet2_phi","jet2_btag",
"jet3_pt","jet3_eta","jet3_phi","jet3_btag",
"jet4_pt","jet4_eta","jet4_phi","jet4_btag",
"m_jj","m_jjj","m_lv","m_jlv",
"m_bb","m_wbb","m_wwbb"
]

df = df.toDF(*columns)

4. Dataset Inspection

In [ ]:
df.show(20)

+-----+------------------+--------------------+--------------------+------------------+--------------------+------------------+--------------------+--------------------+------------------+-------------------+-------------------+--------------------+------------------+------------------+--------------------+--------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+-------------------+------------------+------------------+
|label|         lepton_pt|          lepton_eta|          lepton_phi|    missing_energy|  missing_energy_phi|           jet1_pt|            jet1_eta|            jet1_phi|         jet1_btag|            jet2_pt|           jet2_eta|            jet2_phi|         jet2_btag|           jet3_pt|            jet3_eta|            jet3_phi|        jet3_btag|            jet4_pt|            jet4_eta|            jet4_phi|        jet4_btag|           

In [ ]:
df.printSchema()

root
 |-- label: double (nullable = true)
 |-- lepton_pt: double (nullable = true)
 |-- lepton_eta: double (nullable = true)
 |-- lepton_phi: double (nullable = true)
 |-- missing_energy: double (nullable = true)
 |-- missing_energy_phi: double (nullable = true)
 |-- jet1_pt: double (nullable = true)
 |-- jet1_eta: double (nullable = true)
 |-- jet1_phi: double (nullable = true)
 |-- jet1_btag: double (nullable = true)
 |-- jet2_pt: double (nullable = true)
 |-- jet2_eta: double (nullable = true)
 |-- jet2_phi: double (nullable = true)
 |-- jet2_btag: double (nullable = true)
 |-- jet3_pt: double (nullable = true)
 |-- jet3_eta: double (nullable = true)
 |-- jet3_phi: double (nullable = true)
 |-- jet3_btag: double (nullable = true)
 |-- jet4_pt: double (nullable = true)
 |-- jet4_eta: double (nullable = true)
 |-- jet4_phi: double (nullable = true)
 |-- jet4_btag: double (nullable = true)
 |-- m_jj: double (nullable = true)
 |-- m_jjj: double (nullable = true)
 |-- m_lv: double (nulla

In [ ]:
print("Rows:", df.count())

Rows: 11000000


In [ ]:
print("Columns:", len(df.columns))

Columns: 29


5. File Size Verification

In [ ]:
import os

In [ ]:
size = os.path.getsize("/content/drive/MyDrive/HIGGS.csv.gz")

print("File Size (GB):", round(size/(1024**3),2))

File Size (GB): 2.62


6. Five V Evidence

In [ ]:
print("Partitions:", df.rdd.getNumPartitions())

Partitions: 1


# Task 2: Data Engineering

Missing Values

In [ ]:
from pyspark.sql.functions import col,isnan,when,count

In [ ]:
df.select([
count(
when(
col(c).isNull() | isnan(c),
c
)
).alias(c)
for c in df.columns
]).show()

+-----+---------+----------+----------+--------------+------------------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+----+-----+----+-----+----+-----+------+
|label|lepton_pt|lepton_eta|lepton_phi|missing_energy|missing_energy_phi|jet1_pt|jet1_eta|jet1_phi|jet1_btag|jet2_pt|jet2_eta|jet2_phi|jet2_btag|jet3_pt|jet3_eta|jet3_phi|jet3_btag|jet4_pt|jet4_eta|jet4_phi|jet4_btag|m_jj|m_jjj|m_lv|m_jlv|m_bb|m_wbb|m_wwbb|
+-----+---------+----------+----------+--------------+------------------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+-------+--------+--------+---------+----+-----+----+-----+----+-----+------+
|    0|        0|         0|         0|             0|                 0|      0|       0|       0|        0|      0|       0|       0|        0|      0|       0|       0|        0|      0|       0|       0|        0|   0|    

Duplicate Check

In [ ]:
total_rows = df.count()

unique_rows = df.dropDuplicates().count()

print("Duplicates:",
      total_rows - unique_rows)

Duplicates: 278698


In [ ]:
df = df.dropDuplicates()

print("Rows after removing duplicates:",
      df.count())

Rows after removing duplicates: 10721302


Class Distribution

In [ ]:
df.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|  0.0|4892179|
|  1.0|5829123|
+-----+-------+



Repartition

In [ ]:
df = df.repartition(200)
print(df.rdd.getNumPartitions())

VectorAssembler

In [ ]:
from pyspark.ml.feature import VectorAssembler

In [ ]:
feature_cols = columns[1:]

assembler = VectorAssembler(
inputCols=feature_cols,
outputCol="features_raw"
)

StandardScaler

In [ ]:
from pyspark.ml.feature import StandardScaler

In [ ]:
scaler = StandardScaler(
inputCol="features_raw",
outputCol="features"
)

Pipeline

In [ ]:
from pyspark.ml import Pipeline

In [ ]:
pipeline = Pipeline(
stages=[
assembler,
scaler
]
)

In [ ]:
pipeline_model = pipeline.fit(df)

In [ ]:
processed_df = pipeline_model.transform(df)

In [ ]:
model_df = processed_df.sample(
    False,
    0.05,
    seed=42
)

Train Test Split

In [ ]:
train_df,test_df = model_df.randomSplit(
[0.8,0.2],
seed=42
)

# Task 3: ML Models

## Logistic Regression

In [ ]:
from pyspark.ml.classification import LogisticRegression

In [ ]:
lr = LogisticRegression(
labelCol="label",
featuresCol="features"
)

## Decision Tree

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

In [ ]:
dt = DecisionTreeClassifier(
labelCol="label",
featuresCol="features"
)

## Random Forest

In [ ]:
from pyspark.ml.classification import RandomForestClassifier

In [ ]:
rf = RandomForestClassifier(
labelCol="label",
featuresCol="features"
)

## GBT

In [ ]:
from pyspark.ml.classification import GBTClassifier

In [ ]:
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxBins=32,
    maxDepth=3,
    maxIter=5
)

## Param Grids

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder

In [ ]:
lr_grid = ParamGridBuilder() \
.addGrid(lr.regParam,[0.01]) \
.build()

In [ ]:
dt_grid = ParamGridBuilder() \
.addGrid(dt.maxDepth,[5]) \
.build()

In [ ]:
rf_grid = ParamGridBuilder() \
.addGrid(rf.numTrees,[20]) \
.build()

In [ ]:
gbt_grid = ParamGridBuilder() \
.addGrid(gbt.maxIter,[5,10]) \
.build()

## Evaluator

In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [ ]:
evaluator = BinaryClassificationEvaluator(
labelCol="label"
)

## CrossValidator

In [ ]:
from pyspark.ml.tuning import CrossValidator

In [ ]:
cv_lr = CrossValidator(
estimator=lr,
estimatorParamMaps=lr_grid,
evaluator=evaluator,
numFolds=2
)

In [ ]:
cv_dt = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_grid,
    evaluator=evaluator,
    numFolds=2,
    parallelism=4
)

In [ ]:
cv_rf = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_grid,
    evaluator=evaluator,
    numFolds=2,
    parallelism=4
)

In [ ]:
cv_gbt = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_grid,
    evaluator=evaluator,
    numFolds=2,
    parallelism=4
)

In [ ]:
train_df.cache()

train_df.count()

## Training Time

In [ ]:
import time

In [ ]:
start=time.time()

lr_model=cv_lr.fit(train_df)

end=time.time()

lr_time=round(end-start,2)

print(lr_time)

482.15282225608826


In [ ]:
start = time.time()

dt_model = cv_dt.fit(train_df)

end = time.time()

dt_time = round(end-start,2)

print("Decision Tree Training Time:", dt_time)

Decision Tree Training Time: 216.24


In [ ]:
start = time.time()

rf_model = cv_rf.fit(train_df)

end = time.time()

rf_time = round(end-start,2)

print("Random Forest Training Time:", rf_time)

Random Forest Training Time: 428.52


In [ ]:
start = time.time()

gbt_model = cv_gbt.fit(train_df)

end = time.time()

gbt_time = round(end-start,2)

print("GBT Training Time:", gbt_time)

GBT Training Time: 1998.09


# Task 4: Distributed Computing

Cache

In [ ]:
train_df.cache()

DataFrame[label: double, lepton_pt: double, lepton_eta: double, lepton_phi: double, missing_energy: double, missing_energy_phi: double, jet1_pt: double, jet1_eta: double, jet1_phi: double, jet1_btag: double, jet2_pt: double, jet2_eta: double, jet2_phi: double, jet2_btag: double, jet3_pt: double, jet3_eta: double, jet3_phi: double, jet3_btag: double, jet4_pt: double, jet4_eta: double, jet4_phi: double, jet4_btag: double, m_jj: double, m_jjj: double, m_lv: double, m_jlv: double, m_bb: double, m_wbb: double, m_wwbb: double, features_raw: vector, features: vector]

In [ ]:
train_df.count()

428745

Persist

In [ ]:
from pyspark import StorageLevel

In [ ]:
train_df.persist(
StorageLevel.MEMORY_AND_DISK
)

DataFrame[label: double, lepton_pt: double, lepton_eta: double, lepton_phi: double, missing_energy: double, missing_energy_phi: double, jet1_pt: double, jet1_eta: double, jet1_phi: double, jet1_btag: double, jet2_pt: double, jet2_eta: double, jet2_phi: double, jet2_btag: double, jet3_pt: double, jet3_eta: double, jet3_phi: double, jet3_btag: double, jet4_pt: double, jet4_eta: double, jet4_phi: double, jet4_btag: double, m_jj: double, m_jjj: double, m_lv: double, m_jlv: double, m_bb: double, m_wbb: double, m_wwbb: double, features_raw: vector, features: vector]

Resource Configuration

In [ ]:
for item in spark.sparkContext.getConf().getAll():
    print(item)

('spark.app.name', 'HIGGS_BigData_Project')
('spark.rdd.compress', 'True')
('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K')
('spark.driver.host', '4d69e0c713d9')
('spark.executor.extraJavaOptions', '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --

Repartition Strategy

In [ ]:
train_df = train_df.repartition(200)

Spark UI

In [ ]:
print(
spark.sparkContext.uiWebUrl
)

http://4d69e0c713d9:4040
